# IOAI — 2025 Individual Contest Pixel — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2025"
CLONE = "IOAI-2025"
SUBDIR = "Individual-Contest/Pixel"
WORKDIR = "Individual-Contest/Pixel"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
# 대회 requirements.txt 의 관련 라이브러리 버전으로 정렬하는 constraints(아래 pip 에 적용)
CONSTRAINTS = r'''# IOAI-2025 대회 requirements.txt 에서 추출한 큐레이션 constraints — Colab (과학 패키지 unpin — 세션 numpy 충돌 방지, ML 프레임워크는 대회 핀).
# torch/vllm/xformers/nvidia-cu12 는 제외(환경별 별도 관리).
# 생성: python -m ioai_env constraints
accelerate==1.8.1
bitsandbytes==0.46.1
datasets==3.6.0
einops==0.8.1
ftfy==6.3.1
hf-xet==1.1.5
hf_transfer==0.1.9
huggingface-hub==0.34.2
ipykernel==6.29.5
ipywidgets==7.8.5
jupyter_client==8.6.3
jupyter_core==5.7.2
jupyter_server==2.15.0
jupyterlab==4.3.4
msgspec==0.19.0
notebook==7.3.2
openai==1.90.0
peft==0.16.0
protobuf==5.28.3
pydantic==2.10.3
regex==2024.11.6
safetensors==0.5.3
sentence-transformers==4.1.0
sentencepiece==0.2.0
timm==1.0.16
tokenizers==0.21.2
tqdm==4.67.1
traitlets==5.14.3
transformers==4.54.0
trl==0.19.1
tyro==0.9.26
unsloth==2025.7.8
unsloth_zoo==2025.7.10
'''
open('/content/ioai-constraints.txt', 'w').write(CONSTRAINTS)
!pip install -q -c /content/ioai-constraints.txt transformers datasets accelerate
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# Pixel — 모범답안 2 (개선판: Coarse-to-Fine CLIP 박스탐색)

기존 모범답안(어텐션-마스크 MaskCLIP) = **0.8296**. 이 개선판은 **채점 메트릭을 직접 최적화**합니다:

각 이미지마다 면적 ≤6.25%(=3136px) 직사각형 후보를, 채점기와 똑같이 '박스만 남기고 검정' 처리해 CLIP에 넣어 **타깃 클래스로 분류되며 마진이 가장 큰 박스**를 고릅니다.

**Coarse-to-Fine**: 거친 격자(56×56, stride 24)로 유력 영역 상위 2곳을 찾고, 그 주변만 정밀 탐색 + 종횡비 시도 → 이미지당 후보 ~107개(전수 601개 대비 ~5.6배 적음, ~5배 빠름)로 **점수 ≈0.934 유지**.

> CLIP 분류는 `vision_model`/`text_model` + projection 으로 직접(코사인) — transformers 4.54~5.12 모두 호환.

In [ ]:
import torch, numpy as np, json, os
from transformers import CLIPModel, CLIPProcessor
from datasets import load_dataset
from tqdm.auto import tqdm
DEV='cuda' if torch.cuda.is_available() else 'cpu'
MODEL_PATH='openai/clip-vit-large-patch14'
model=CLIPModel.from_pretrained(MODEL_PATH).to(DEV).eval()
proc=CLIPProcessor.from_pretrained(MODEL_PATH)

In [ ]:
# 데이터(로컬 사이트=patches 가 로컬 스냅샷으로, Colab=HF 다운로드)
test_dataset=load_dataset('IOAI-official/IOAI-2025-Pixel-test', data_dir='data', split='test')
train_dataset=load_dataset('IOAI-official/IOAI-2025-Pixel-train', data_dir='data', split='train')
classes=sorted(set(train_dataset['name']))+['other']
text_inputs=proc(text=classes, return_tensors='pt', padding=True).to(DEV)
# 텍스트 임베딩 1회 precompute (버전 무관: text_model + text_projection)
with torch.no_grad():
    _to=model.text_model(input_ids=text_inputs['input_ids'], attention_mask=text_inputs['attention_mask'])
    tfeat=model.text_projection(_to.pooler_output); tfeat=tfeat/tfeat.norm(dim=-1,keepdim=True)
print('classes:', classes)

In [ ]:
# 검정(0) 정규화 후 직접 마스킹 → vision_model 로 이미지 임베딩(버전 무관) → 코사인 logits
MEAN=torch.tensor([0.48145466,0.4578275,0.40821073],device=DEV).view(3,1,1)
STD=torch.tensor([0.26862954,0.26130258,0.27577711],device=DEV).view(3,1,1)
BLACK=((0-MEAN)/STD).expand(3,224,224)
def norm_img(pil):
    a=torch.from_numpy(np.asarray(pil.convert('RGB'),dtype=np.float32)).permute(2,0,1).to(DEV)/255.0
    return (a-MEAN)/STD
def logits_of(boxes, ni):
    mb=BLACK.unsqueeze(0).repeat(len(boxes),1,1,1).clone()
    for j,(t,l,b,r) in enumerate(boxes): mb[j,:,t:b,l:r]=ni[:,t:b,l:r]
    io=model.vision_model(pixel_values=mb); f=model.visual_projection(io.pooler_output)
    f=f/f.norm(dim=-1,keepdim=True); return f@tfeat.T          # [B, classes]
SIZES=[(56,56),(44,71),(71,44),(48,65),(65,48)]   # 모두 ≤3136 px²
COARSE=[(t,l,t+56,l+56) for t in range(0,169,24) for l in range(0,169,24)]  # 56x56 stride24 → 49

In [ ]:
def _scores(boxes, lg, target):
    pred=lg.argmax(1); tgt=lg[:,target]; oth=lg.clone(); oth[:,target]=-1e9; mg=tgt-oth.max(1).values
    return pred, tgt, mg
def _best(boxes, lg, target):
    pred,tgt,mg=_scores(boxes,lg,target); best=None; bm=-1e9
    for j in range(len(boxes)):
        if int(pred[j])==target and float(mg[j])>bm: bm=float(mg[j]); best=boxes[j]
    if best is None:
        for j in range(len(boxes)):
            if float(tgt[j])>bm: bm=float(tgt[j]); best=boxes[j]
    return best
def best_box(ni, target, topk=2):
    lg=logits_of(COARSE, ni); pred,tgt,mg=_scores(COARSE,lg,target)
    key=[(float(mg[j]) if int(pred[j])==target else -1e6+float(tgt[j])) for j in range(len(COARSE))]
    top=sorted(range(len(COARSE)), key=lambda j:key[j], reverse=True)[:topk]   # 유력 영역 상위 topk
    fine=set()
    for j in top:
        t0,l0,_,_=COARSE[j]
        for dt in (-16,-8,0,8,16):
            for dl in (-16,-8,0,8,16):
                t,l=t0+dt,l0+dl
                if 0<=t<=168 and 0<=l<=168: fine.add((t,l,t+56,l+56))   # 정밀 위치
        for (h,w) in SIZES:                                             # 그 영역 종횡비
            if t0+h<=224 and l0+w<=224: fine.add((t0,l0,t0+h,l0+w))
    fine=list(fine); return _best(fine, logits_of(fine, ni), target)

In [ ]:
# 전체 테스트셋 coarse-to-fine 박스탐색 → submission.jsonl
sub=[]
with torch.no_grad():
    for item in tqdm(test_dataset, desc='c2f box search'):
        ni=norm_img(item['image'])
        target=int(logits_of([(0,0,224,224)], ni)[0].argmax())   # 전체이미지 CLIP 예측=타깃
        t,l,b,r=best_box(ni, target)
        sub.append({'idx':item['idx'],'coordinates':[[int(t),int(l)],[int(b),int(r)]]})
with open('submission.jsonl','w') as f:
    for r in sub: f.write(json.dumps(r)+'\n')
print('submission.jsonl ready (%d masks)'%len(sub))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.jsonl']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)